# Feature Extraction & Deposit

In this notebook we define an example of creating a database from a .csv and perform feature extraction of audio files. The APIs are auto-generated from the Swagger Endpoint documentations using [`generate.sh`](https://gitlab.phaidra.org/fair-data-austria-db-repository/fda-docs/-/blob/master/swagger/generate.sh). Steps we perform:

  1. Download a music file from a public repository
  2. Perform feature extraction
  3. Obtain an authentication token
  4. Create a mariadb container
  5. Start the mariadb container
  6. Create a database within the mariadb container
  7. Import the feature .csv (manually)

Please create an account at [http://localhost:3000/register](http://localhost:3000/register) with `user:user` before executing.

In [13]:
import uuid
import time
import re
import requests as rq
from api_authentication.api.authentication_endpoint_api import AuthenticationEndpointApi
from api_authentication.api.user_endpoint_api import UserEndpointApi
from api_container.api.container_endpoint_api import ContainerEndpointApi
from api_database.api.container_database_endpoint_api import ContainerDatabaseEndpointApi
from api_table.api.table_endpoint_api import TableEndpointApi
from api_query.api.table_data_endpoint_api import TableDataEndpointApi
from api_query.api.query_endpoint_api import QueryEndpointApi
from api_identifier.api.identifier_endpoint_api import IdentifierEndpointApi

authentication = AuthenticationEndpointApi()
user = UserEndpointApi()
container = ContainerEndpointApi()
database = ContainerDatabaseEndpointApi()
table = TableEndpointApi()
query = QueryEndpointApi()
data = TableDataEndpointApi()
identifier = IdentifierEndpointApi()

url = "https://test.researchdata.tuwien.ac.at/records/vqpbr-5b889"
audio = "http://localhost:8000/v1/audio"

In [14]:
response = authentication.authenticate_user1({
    "username": "user",
    "password": "user"
})
user_id = response.id
token = response.token
container.api_client.default_headers = {"Authorization": "Bearer " + token}
database.api_client.default_headers = {"Authorization": "Bearer " + token}
table.api_client.default_headers = {"Authorization": "Bearer " + token}
data.api_client.default_headers = {"Authorization": "Bearer " + token}
query.api_client.default_headers = {"Authorization": "Bearer " + token}
identifier.api_client.default_headers = {"Authorization": "Bearer " + token}
user.api_client.default_headers = {"Authorization": "Bearer " + token}

In [15]:
response = user.update({
    "firstname": "Martin",
    "lastname": "Weise",
    "titles_before": "DI"
}, user_id)
print(response)

{'authorities': [{'authority': 'ROLE_RESEARCHER'}],
 'containers': None,
 'databases': None,
 'email': 'martin.weise@tuwien.ac.at',
 'email_verified': False,
 'firstname': 'Martin',
 'id': 2,
 'identifiers': None,
 'lastname': 'Weise',
 'titles_after': None,
 'titles_before': 'DI',
 'username': 'user'}


In [16]:
response = container.create1({
    "name": "ethmusmir " + str(uuid.uuid1()),
    "repository": "mariadb",
    "tag": "10.5"
})
container_id = response.id
print(response)

ApiException: (500)
Reason: Internal Server Error
HTTP response headers: HTTPHeaderDict({'Content-Type': 'text/html;charset=utf-8', 'Content-Language': 'en', 'Content-Length': '455', 'Date': 'Sat, 09 Jul 2022 15:07:39 GMT'})
HTTP response body: b'<!doctype html><html lang="en"><head><title>HTTP Status 500 \xe2\x80\x93 Internal Server Error</title><style type="text/css">body {font-family:Tahoma,Arial,sans-serif;} h1, h2, h3, b {color:white;background-color:#525D76;} h1 {font-size:22px;} h2 {font-size:16px;} h3 {font-size:14px;} p {font-size:12px;} a {color:black;} .line {height:1px;background-color:#525D76;border:none;}</style></head><body><h1>HTTP Status 500 \xe2\x80\x93 Internal Server Error</h1></body></html>'


In [ ]:
response = container.modify({
    "action": "start"
}, container_id)
time.sleep(5)
print(response)

In [ ]:
response = database.create({
    "name": "ethmusmir " + str(uuid.uuid1()),
    "description": "Feature Vectors extracted with the EthMusMIR Analysis Server https://github.com/ketchupok/ethmusmir applied on a remixed recording of the SeFiRe field recordings dataset https://github.com/matijama/field-recording-db",
    "is_public": True
}, container_id)
database_id = response.id
print(response)

In [ ]:
response = table.create({
    "name": "Feature Extraction",
    "description": "SeFiRe",
    "columns": [{
        "name": "Content",
        "type": "STRING",
        "unique": False,
        "primary_key": False,
        "null_allowed": True,
    }, {
        "name": "Start",
        "type": "NUMBER",
        "unique": False,
        "primary_key": False,
        "null_allowed": True,
    }, {
        "name": "Duration",
        "type": "NUMBER",
        "unique": False,
        "primary_key": False,
        "null_allowed": True,
    }]
}, container_id, database_id)
table_id = response.id
print(response)

In [ ]:
host = re.findall("^https?:\/\/([a-z0-9\.]+)", url)[0]
id = re.findall("/([a-z0-9-]+)$", url)[0]

response = rq.get("https://" + host + "/api/records/" + id + "/files")
record = response.json()

In [ ]:
for file in record["entries"]:
    print("... save file contents from", file["links"]["content"])
    wav = rq.get(file["links"]["content"])
    filename = "/tmp/" + file["key"]
    open(filename, "wb").write(wav.content)
    with open(filename, "rb") as f:
        payload = f.read()
        res = rq.post(audio, data=payload, headers={"Content-Type": "audio/wav"})
        print("... extracted", res.json())
        for part in res.json()["track"]["parts"]:
            response = data.insert({"data": part}, container_id, database_id, table_id)
            print(response)

In [ ]:
response = query.execute({"statement": "SELECT `content`, `start`, `duration` FROM `feature_extraction`"}, container_id, database_id)
query_id = response.id
print(response)

In [ ]:
response = identifier.create({
    "qid": query_id,
    "title": "title",
    "description": "description",
    "visibility": "everyone",
    "creators": [{
      "name": "Weise, Martin",
      "affiliation": "TU Wien",
      "orcid": "0000-0003-4216-302X"
    }, {
      "name": "Rauber, Andreas",
      "affiliation": "TU Wien",
      "orcid": "0000-0002-9272-6225"
    }],
    "publication_year": 2022,
    "related_identifiers": [{
      "value": url,
      "type": "URL",
      "relation": "IsCitedBy"
    }]
}, token, container_id, database_id)
identifier_id = response.id
print(response)